<a href="https://colab.research.google.com/github/hwasun-zip/Catalog-Triage/blob/main/Catalog_Triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 핵심 질문: 수많은 상점의 카탈로그 문제 중, 어떤 문제를 먼저 해결해야 비즈니스에 가장 큰 효과를 낼 수 있을까?
# 파이프라인: 이상 탐지 → 원인 분석 → 비즈니스 영향도 분석 → 우선순위 도출(P1/P2/P3) → 개선안 제안

# 데이터 출처 및 한계: 실제 운영 데이터 접근 권한이 없어 공개 데이터셋(Uber Eats USA Restaurants and Menus, Kaggle)을 기반으로 한다.
# 최소주문금액, 이미지 품질, 전환 로그 등 없는 필드는 현실적인 분포 가정 하에 합성(synthetic)했으며 각 셀에서 명시한다.
# Business Impact 분석은 인과관계가 아닌 '가상 시나리오 하의 그룹 간 차이 비교'로 해석해야 하며, 실제 서비스라면 A/B 테스트/DID 등 인과추론이 추가로 필요하다.

In [2]:
# 라이브러리 설치&import
!pip install -q duckdb kagglehub

import duckdb # DataFrame 위에서 SQL(CTE, Window Function 등)을 돌리기 위한 엔진
import pandas as pd # 데이터 정제, 합성 필드 생성용
import numpy as np # 데이터 정제, 합성 필드 생성용
import kagglehub # Kaggle 데이터셋을 Colab으로 바로 받아오기 위한 라이브러리
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

print('duckdb version:', duckdb.__version__)
print('pandas version:', pd.__version__)

duckdb version: 1.3.2
pandas version: 2.2.3


In [3]:
# Uber Eats USA Restaurants and Menus (Kaggle) - 6.3만+ 상점, 500만+ 메뉴 아이템

if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('kaggle.json 파일을 업로드해주세요 (Kaggle > Settings > API > Create New Token)')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fn in uploaded.keys():
        os.rename(fn, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

dataset_path = kagglehub.dataset_download('ahmedshahriarsakib/uber-eats-usa-restaurants-menus')
print('다운로드 완료:', dataset_path)
print(os.listdir(dataset_path))

kaggle.json 파일을 업로드해주세요 (Kaggle > Settings > API > Create New Token)


Saving kaggle (1).json to kaggle (1).json


100%|██████████| 193M/193M [00:01<00:00, 110MB/s] 

Extracting files...


다운로드 완료: /root/.cache/kagglehub/datasets/ahmedshahriarsakib/uber-eats-usa-restaurants-menus/versions/12
['restaurant-menus.csv', 'restaurants.csv']


In [4]:
# restaurants.csv: 상점 단위 정보 (카테고리, 지역, 평점 등)
# restaurant-menus.csv: 메뉴 아이템 단위 정보 (상점 id, 메뉴명, 가격 등)

restaurants_raw = pd.read_csv(f'{dataset_path}/restaurants.csv')
menus_raw = pd.read_csv(f'{dataset_path}/restaurant-menus.csv')

print('restaurants shape:', restaurants_raw.shape)
print('menus shape:', menus_raw.shape)
print()
print('--- restaurants columns ---')
print(restaurants_raw.columns.tolist())
print()
print('--- menus columns ---')
print(menus_raw.columns.tolist())

restaurants shape: (63469, 11)
menus shape: (5117217, 5)

--- restaurants columns ---
['id', 'position', 'name', 'score', 'ratings', 'category', 'price_range', 'full_address', 'zip_code', 'lat', 'lng']

--- menus columns ---
['restaurant_id', 'category', 'name', 'description', 'price']


In [5]:
# 탐색적 데이터 분석(EDA)
# 발견 1: price가 문자열
# 발견 2: price_range는 거의 $/$$뿐
# 발견 3: category가 다중 태그 문자열
# 발견 4: score/ratings 결측치 많음
# 발견 5: 마이너스 가격 존재
# 발견 6: '음식'이 아닌 상품도 섞여있음

In [6]:
# 상점 데이터 기초 탐색
print('--- price_range 분포 ---')
print(restaurants_raw['price_range'].value_counts(dropna=False))
print()
print('--- category 분포 (상위 15개) ---')
print(restaurants_raw['category'].value_counts(dropna=False).head(15))
print()
print('--- score/ratings 기초 통계 ---')
print(restaurants_raw[['score', 'ratings']].describe())

--- price_range 분포 ---
price_range
$                    37637
$$                   14952
NaN                  10617
$$$                    237
$$$$                    25
$$$$$$$$$$$$$$$$$        1
Name: count, dtype: int64

--- category 분포 (상위 15개) ---
category
Burgers, American, Sandwiches                           2439
Mexican, Latin American, New Mexican                    2273
Fast Food, Sandwich, American                           1196
American, Burgers, Fast Food                             995
Pizza, American, Italian                                 961
Burritos, Fast Food, Mexican                             637
American, Burgers, Sandwiches                            632
American, burger, Fast Food                              602
Coffee and Tea, American, Breakfast and Brunch           585
American, Fast Food, Burgers                             556
Chinese, Asian, Asian Fusion                             490
Pharmacy, Convenience, Everyday Essentials, Baby         477
Americ

In [7]:
# 메뉴 데이터 기초 탐색
# price가 "12.5 USD" 형태의 문자열이라 숫자로 변환 필요
menus_raw['price_numeric'] = (
    menus_raw['price']
    .str.replace(' USD', '', regex=False)
    .astype(float)
)

print('--- price_numeric 기초 통계 ---')
print(menus_raw['price_numeric'].describe())
print()
print('--- 0원 메뉴 개수 ---')
print((menus_raw['price_numeric'] == 0).sum())
print()
print('--- 극단적 고가 메뉴 미리보기 (200달러 초과) ---')
print(menus_raw[menus_raw['price_numeric'] > 200][['name', 'price_numeric']].head(10))
print()
print('--- price가 null인 경우 ---')
print(menus_raw['price_numeric'].isna().sum())

--- price_numeric 기초 통계 ---
count    5.117217e+06
mean     1.036867e+01
std      1.395465e+01
min     -7.050000e+00
25%      3.900000e+00
50%      7.990000e+00
75%      1.299000e+01
max      1.395000e+03
Name: price_numeric, dtype: float64

--- 0원 메뉴 개수 ---
174294

--- 극단적 고가 메뉴 미리보기 (200달러 초과) ---
                                                    name  price_numeric
34116  Therapedic® TruCool® 3-Inch Serene Foam King M...         367.99
34336  Dyson Air TP01 Multiplier 40-Inch Bladeless To...         299.99
34348  Shark® Wandvac System™ Cordless Ultra-Light St...         259.99
34471  Chicco® Fit4® Adapt 4-in-1 Convertible Car Sea...         389.99
34472     Graco® SlimFit3 LX 3-in-1 Car Seat in Stanford         279.99
34474      Chicco MyFit® Harness + Booster Seat in Notte         209.99
34481  Graco® 4Ever® DLX 4-in-1 Convertible Car Seat ...         329.99
34484  KNOX® Convertible Car Seat by UPPAbaby® in Jordan         399.99
34486   Maxi-Cosi® Coral™ XP Infant Car Seat in Grap

In [8]:
# category 대표 태그 추출 (첫 번째 태그를 대표 카테고리로 사용)
restaurants_raw['main_category'] = restaurants_raw['category'].str.split(',').str[0].str.strip()

print('--- 대표 카테고리 분포 (상위 15개) ---')
print(restaurants_raw['main_category'].value_counts(dropna=False).head(15))

--- 대표 카테고리 분포 (상위 15개) ---
main_category
American                12302
Mexican                  5141
Burgers                  4507
Pizza                    3113
Breakfast and Brunch     1859
Fast Food                1820
Everyday Essentials      1574
Desserts                 1550
Asian                    1546
Chinese                  1543
Italian                  1381
Indian                   1344
Sandwich                 1236
Seafood                  1198
Bakery                   1043
Name: count, dtype: int64


In [9]:
# 대표 카테고리 전체 개수 및 나머지 목록 확인 (비음식 카테고리 찾기용)
print('--- 대표 카테고리 개수(unique) ---')
print(restaurants_raw['main_category'].nunique())
print()
print('--- 대표 카테고리 전체 목록 (알파벳순) ---')
print(sorted(restaurants_raw['main_category'].dropna().unique().tolist()))

--- 대표 카테고리 개수(unique) ---
264

--- 대표 카테고리 전체 목록 (알파벳순) ---
['*New', 'AAPI-owned', 'Adult', 'Affordable Meals', 'Afghan', 'African', 'African: Ethiopian', 'African: Other', 'Alcohol', 'Allergy Friendly', 'Ameican', 'American', 'American (New)', 'American (Traditional)', 'Anglo-Indian', 'Appetizers', 'Arabian', 'Arepa', 'Argentinian', 'Asian', 'Asian Fusion', 'Asian-owned', 'Asian: Other', 'Australian', 'Authentic Italian', 'Açaí', 'BBQ', 'Bagels', 'Bakery', 'Bangladeshi', 'Bar / Pub Food', 'Bar Food', 'Beef Noodles', 'Beer', 'Bento', 'Biryani', 'Bistro', 'Black-owned', 'Bowls', 'Brazilian', 'Breakfast', 'Breakfast &amp; Brunch', 'Breakfast and Brunch', 'British', 'Bubble Tea', 'Burgers', 'Burritos', 'Cafe', 'Cajun', 'Cajun / Creole', 'Cambodian', 'Canadian', 'Candy', 'Cantonese', 'Caribbean', 'Cheesesteak', 'Chicken', 'Chicken Strips', 'Chilean', 'Chinese', 'Chinese Food', 'Chinese: Cantonese', 'Chinese: Hot Pot', 'Chinese: Noodles &amp; Dumplings', 'Chinese: Other', 'Chinese: Sichuan

In [10]:
# 음수 가격 메뉴 확인
print('--- 음수 가격 메뉴 개수 ---')
print((menus_raw['price_numeric'] < 0).sum())
print()
print('--- 음수 가격 미리보기 ---')
print(menus_raw[menus_raw['price_numeric'] < 0][['restaurant_id', 'name', 'price_numeric']].head(10))

--- 음수 가격 메뉴 개수 ---
3

--- 음수 가격 미리보기 ---
         restaurant_id                 name  price_numeric
660059            7658      Chicken en Mole          -7.05
660061            7658  Pollo a la Ranchera          -7.05
3658845          44311          Apple Juice          -2.50


In [11]:
# 데이터 정제 (비음식 필터링, price 파싱)
# 비음식(리테일/편의점/주류/잡화) 카테고리 제외 리스트
non_food_categories = [
    'Alcohol', 'Beer', 'Wine', 'Liquor Stores',
    'Convenience', 'Convenience Store with Alcohol', 'Everyday Essentials',
    'Grocery', 'Retail', 'OrganicProducts',
    'Pharmacy', 'Farmacia', 'Personal Care', 'Home &amp; Personal Care',
    'Pet Shop', 'pet supplies',
    'Flowers', 'florist', 'Gift Store', 'bookstore', 'chocolatier'
]

food_restaurants = restaurants_raw[~restaurants_raw['main_category'].isin(non_food_categories)].copy()

print('필터링 전 상점 수:', len(restaurants_raw))
print('필터링 후 상점 수:', len(food_restaurants))
print('제외된 상점 수:', len(restaurants_raw) - len(food_restaurants))
print()
print('--- 제외된 카테고리별 상점 수 ---')
print(restaurants_raw[restaurants_raw['main_category'].isin(non_food_categories)]['main_category'].value_counts())

필터링 전 상점 수: 63469
필터링 후 상점 수: 59521
제외된 상점 수: 3948

--- 제외된 카테고리별 상점 수 ---
main_category
Everyday Essentials               1574
Convenience                        693
Pharmacy                           529
Alcohol                            496
Grocery                            289
Flowers                            173
Retail                             123
Gift Store                          23
Liquor Stores                       12
Home &amp; Personal Care             6
Wine                                 5
chocolatier                          5
florist                              4
Pet Shop                             3
Beer                                 3
Convenience Store with Alcohol       3
pet supplies                         3
OrganicProducts                      1
bookstore                            1
Farmacia                             1
Personal Care                        1
Name: count, dtype: int64


In [12]:
# 음식점 상점에 해당하는 메뉴만 남기기 (restaurant_id 기준 조인)
food_menus = menus_raw[menus_raw['restaurant_id'].isin(food_restaurants['id'])].copy()

print('필터링 전 메뉴 수:', len(menus_raw))
print('필터링 후 메뉴 수:', len(food_menus))

필터링 전 메뉴 수: 5117217
필터링 후 메뉴 수: 4236129


In [13]:
# 최소 주문금액 로직
np.random.seed(42)  # 재현 가능성을 위해 시드 고정

# 1. 상점별 '메인 메뉴 평균가' = 가격 상위 30~70% 구간의 평균 (Trimmed Mean)
#    사이드/음료(저가) + 세트메뉴(고가) 양극단을 배제해 착시를 줄임
def trimmed_mean_price(prices):
    if len(prices) < 3:
        return prices.mean()
    q30, q70 = prices.quantile([0.3, 0.7])
    trimmed = prices[(prices >= q30) & (prices <= q70)]
    return trimmed.mean() if len(trimmed) > 0 else prices.mean()

main_menu_avg = food_menus.groupby('restaurant_id')['price_numeric'].apply(trimmed_mean_price).rename('main_menu_avg_price')
food_restaurants = food_restaurants.merge(main_menu_avg, left_on='id', right_index=True, how='left')

# 2. 정상 배수(1.5~2.0배, "메인메뉴+사이드 하나" 수준)로 최소주문금액 생성
normal_multiplier = np.random.uniform(1.5, 2.0, size=len(food_restaurants))
food_restaurants['min_order_amount'] = (food_restaurants['main_menu_avg_price'] * normal_multiplier).round(2)

# 3. 일부 상점(약 5%)에 의도적 이상치 주입 - 나중에 탐지 정확도(recall) 검증용 ground truth로 활용
outlier_mask = np.random.rand(len(food_restaurants)) < 0.05
outlier_type = np.random.rand(outlier_mask.sum()) < 0.5

outlier_idx = food_restaurants[outlier_mask].index
high_idx = outlier_idx[outlier_type]      # 과도하게 높은 케이스 (P1 후보)
low_idx = outlier_idx[~outlier_type]       # 비정상적으로 낮은 케이스 (P3 후보)

food_restaurants.loc[high_idx, 'min_order_amount'] = (
    food_restaurants.loc[high_idx, 'main_menu_avg_price'] * np.random.uniform(3.0, 5.0, size=len(high_idx))
).round(2)

food_restaurants.loc[low_idx, 'min_order_amount'] = (
    food_restaurants.loc[low_idx, 'main_menu_avg_price'] * np.random.uniform(0.2, 0.7, size=len(low_idx))
).round(2)

# 4. 검증용 라벨 저장 (나중에 SQL 탐지 결과와 비교할 ground truth)
food_restaurants['is_synthetic_outlier'] = outlier_mask
food_restaurants['synthetic_outlier_type'] = np.select(
    [food_restaurants['id'].isin(food_restaurants.loc[high_idx, 'id']),
     food_restaurants['id'].isin(food_restaurants.loc[low_idx, 'id'])],
    ['high', 'low'],
    default='normal'
)

print('--- min_order_amount 기초 통계 ---')
print(food_restaurants['min_order_amount'].describe())
print()
print('--- 결측(메뉴 0건 상점) ---')
print(food_restaurants['main_menu_avg_price'].isna().sum())
print()
print('--- 주입된 이상치 개수 ---')
print(food_restaurants['synthetic_outlier_type'].value_counts())

--- min_order_amount 기초 통계 ---
count    59376.000000
mean        15.699277
std          9.665552
min          0.000000
25%          9.530000
50%         14.660000
75%         20.190000
max        329.180000
Name: min_order_amount, dtype: float64

--- 결측(메뉴 0건 상점) ---
145

--- 주입된 이상치 개수 ---
synthetic_outlier_type
normal    56572
high       1501
low        1448
Name: count, dtype: int64


In [14]:
# 1. 메뉴가 없는 상점(145건) 제외 -> 최종 분석 대상 확정
analysis_restaurants = food_restaurants[food_restaurants['main_menu_avg_price'].notna()].copy()
print('최종 분석 대상 상점 수:', len(analysis_restaurants))

# 2. main_menu_avg_price가 0원인 상점 확인 (가격 데이터 오류가 전이된 케이스)
zero_price_stores = analysis_restaurants[analysis_restaurants['main_menu_avg_price'] == 0]
print()
print('메인메뉴 평균가 0원인 상점 수:', len(zero_price_stores))
print(zero_price_stores[['id', 'name', 'main_menu_avg_price', 'min_order_amount']].head(10))

최종 분석 대상 상점 수: 59376

메인메뉴 평균가 0원인 상점 수: 695
      id                                              name  main_menu_avg_price  min_order_amount
138  139                           Dunkin (300 Commons Dr)                  0.0               0.0
312  313                  Bayleaf Authentic Indian Cuisine                  0.0               0.0
429  430                    Dunkin' (2536 Helena Rd Ste D)                  0.0               0.0
487  488                           Dunkin' (2419 Acton Rd)                  0.0               0.0
542  543              Baskin-Robbins (3064 Ross Clark Cir)                  0.0               0.0
582  583                     Dunkin' (4185 Montgomery Hwy)                  0.0               0.0
746  747                       Dunkin' (103 Brookridge Dr)                  0.0               0.0
828  829                  Baskin-Robbins (1024 6th Ave Se)                  0.0               0.0
943  944                 Dunkin' (2785 Carl T Jones Dr Se)               

In [15]:
# '가격정보 미기재' 플래그를 실제 컬럼으로 저장
analysis_restaurants['price_data_status'] = np.where(
    analysis_restaurants['main_menu_avg_price'] == 0,
    '가격정보 미기재',
    '정상'
)

print(analysis_restaurants['price_data_status'].value_counts())

price_data_status
정상          58681
가격정보 미기재      695
Name: count, dtype: int64


In [16]:
# DuckDB에 분석 대상 테이블 등록 (컬럼 추가 완료된 최신 상태로 등록)
con = duckdb.connect()
con.register('restaurants', analysis_restaurants)
con.register('menus', food_menus)

print(con.execute("SELECT COUNT(*) AS restaurant_count FROM restaurants").fetchdf())
print(con.execute("SELECT COUNT(*) AS menu_count FROM menus").fetchdf())
print()
print(con.execute("SELECT price_data_status, COUNT(*) AS store_count FROM restaurants GROUP BY price_data_status").fetchdf())

   restaurant_count
0             59376
   menu_count
0     4236129

  price_data_status  store_count
0                정상        58681
1          가격정보 미기재          695


In [17]:
# SQL 이상탐지 — 첫 번째 대상: 가격 이상치
price_outliers_query = """
WITH menu_with_category AS (
    -- 메뉴에 상점의 대표 카테고리를 붙임
    SELECT
        m.restaurant_id,
        m.name AS menu_name,
        m.price_numeric,
        r.name AS restaurant_name,
        r.main_category
    FROM menus m
    JOIN restaurants r ON m.restaurant_id = r.id
    WHERE m.price_numeric > 0  -- 0원/음수는 별도 이슈로 다루므로 여기선 제외
),
category_stats AS (
    -- 카테고리별 가격 분포 통계를 Window Function으로 계산
    SELECT
        *,
        AVG(price_numeric) OVER (PARTITION BY main_category) AS category_avg_price,
        STDDEV(price_numeric) OVER (PARTITION BY main_category) AS category_std_price,
        COUNT(*) OVER (PARTITION BY main_category) AS category_menu_count
    FROM menu_with_category
),
z_score_flagged AS (
    -- Z-score 계산 및 이상치 플래깅 (표본이 너무 적은 카테고리는 제외)
    SELECT
        *,
        CASE
            WHEN category_std_price > 0
            THEN (price_numeric - category_avg_price) / category_std_price
            ELSE 0
        END AS price_z_score
    FROM category_stats
    WHERE category_menu_count >= 30  -- 통계적 신뢰를 위해 표본 30건 이상 카테고리만
)
SELECT
    restaurant_id,
    restaurant_name,
    menu_name,
    main_category,
    price_numeric,
    ROUND(category_avg_price, 2) AS category_avg_price,
    ROUND(price_z_score, 2) AS price_z_score
FROM z_score_flagged
WHERE ABS(price_z_score) > 3
ORDER BY ABS(price_z_score) DESC
LIMIT 20
"""

price_outliers = con.execute(price_outliers_query).fetchdf()
print('가격 이상치로 탐지된 메뉴 미리보기 (상위 20건):')
print(price_outliers)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

가격 이상치로 탐지된 메뉴 미리보기 (상위 20건):
    restaurant_id                           restaurant_name                                          menu_name                  main_category  \
0           54106                             Fire Ass Thai                               Spicy Massaman Curry                   Asian Fusion   
1           19112              Tsim Yung Chinese Restaurant                               C 7. Moo Goo Gai Pan                        Chinese   
2           41586                          Taqueria Chapala                                      Mojarra Frita                        Mexican   
3           36135                         palios pizza cafe                                        Greek Bread                        Italian   
4           50119                          Nusr-Et (Dallas)                      Gold Istanbul Steak (10.5 oz)                       American   
5           29458    Rabble-Rouser Chocolate &amp; Craft Co                                     The 

In [18]:
# 전체 이상치 규모 확인
count_query = """
WITH menu_with_category AS (
    SELECT m.restaurant_id, m.price_numeric, r.main_category
    FROM menus m JOIN restaurants r ON m.restaurant_id = r.id
    WHERE m.price_numeric > 0
),
category_stats AS (
    SELECT *,
        AVG(price_numeric) OVER (PARTITION BY main_category) AS category_avg_price,
        STDDEV(price_numeric) OVER (PARTITION BY main_category) AS category_std_price,
        COUNT(*) OVER (PARTITION BY main_category) AS category_menu_count
    FROM menu_with_category
)
SELECT COUNT(*) AS total_price_outliers
FROM category_stats
WHERE category_menu_count >= 30
  AND category_std_price > 0
  AND ABS((price_numeric - category_avg_price) / category_std_price) > 3
"""
print(con.execute(count_query).fetchdf())

   total_price_outliers
0                 60240


In [19]:
# 소수점 오류 의심
decimal_error_query = """
WITH menu_with_category AS (
    SELECT
        m.restaurant_id,
        m.name AS menu_name,
        m.price_numeric,
        r.name AS restaurant_name,
        r.main_category
    FROM menus m
    JOIN restaurants r ON m.restaurant_id = r.id
    WHERE m.price_numeric > 0
),
category_stats AS (
    SELECT
        *,
        AVG(price_numeric) OVER (PARTITION BY main_category) AS category_avg_price,
        STDDEV(price_numeric) OVER (PARTITION BY main_category) AS category_std_price,
        COUNT(*) OVER (PARTITION BY main_category) AS category_menu_count
    FROM menu_with_category
),
z_score_flagged AS (
    SELECT
        *,
        CASE
            WHEN category_std_price > 0
            THEN (price_numeric - category_avg_price) / category_std_price
            ELSE 0
        END AS price_z_score
    FROM category_stats
    WHERE category_menu_count >= 30
),
outliers AS (
    SELECT *
    FROM z_score_flagged
    WHERE ABS(price_z_score) > 3
)
SELECT
    restaurant_id,
    restaurant_name,
    menu_name,
    main_category,
    price_numeric,
    ROUND(category_avg_price, 2) AS category_avg_price,
    ROUND(price_z_score, 2) AS price_z_score,
    -- 소수점 오류 의심: 100으로 나누거나 10으로 나눴을 때 카테고리 평균과 가까워지는가
    CASE
        WHEN ABS(price_numeric / 100 - category_avg_price) / NULLIF(category_avg_price, 0) < 0.3
            THEN '소수점 오류 의심 (÷100)'
        WHEN ABS(price_numeric / 10 - category_avg_price) / NULLIF(category_avg_price, 0) < 0.3
            THEN '소수점 오류 의심 (÷10)'
        ELSE '검토 필요 (원인 불명)'
    END AS suspected_cause
FROM outliers
ORDER BY ABS(price_z_score) DESC
"""

decimal_check = con.execute(decimal_error_query).fetchdf()
print('원인별 분류 결과:')
print(decimal_check['suspected_cause'].value_counts())
print()
print('상위 20건 미리보기:')
print(decimal_check.head(20)[['restaurant_name', 'menu_name', 'price_numeric', 'category_avg_price', 'suspected_cause']])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

원인별 분류 결과:
suspected_cause
검토 필요 (원인 불명)       49740
소수점 오류 의심 (÷10)     10495
소수점 오류 의심 (÷100)        5
Name: count, dtype: int64

상위 20건 미리보기:
                             restaurant_name                                          menu_name  price_numeric  category_avg_price   suspected_cause
0                              Fire Ass Thai                               Spicy Massaman Curry        1395.00               11.98  소수점 오류 의심 (÷100)
1               Tsim Yung Chinese Restaurant                               C 7. Moo Goo Gai Pan         925.00               11.97  소수점 오류 의심 (÷100)
2                           Taqueria Chapala                                      Mojarra Frita        1049.00                9.87  소수점 오류 의심 (÷100)
3                          palios pizza cafe                                        Greek Bread         599.00               13.38     검토 필요 (원인 불명)
4                           Nusr-Et (Dallas)                      Gold Istanbul Steak (10.5 oz)         590.00

In [20]:
# 절대 배수 조건 추가
final_price_outliers_query = """
WITH menu_with_category AS (
    SELECT
        m.restaurant_id,
        m.name AS menu_name,
        m.price_numeric,
        r.name AS restaurant_name,
        r.main_category
    FROM menus m
    JOIN restaurants r ON m.restaurant_id = r.id
    WHERE m.price_numeric > 0
),
category_stats AS (
    SELECT
        *,
        AVG(price_numeric) OVER (PARTITION BY main_category) AS category_avg_price,
        STDDEV(price_numeric) OVER (PARTITION BY main_category) AS category_std_price,
        COUNT(*) OVER (PARTITION BY main_category) AS category_menu_count
    FROM menu_with_category
),
z_score_flagged AS (
    SELECT
        *,
        CASE
            WHEN category_std_price > 0
            THEN (price_numeric - category_avg_price) / category_std_price
            ELSE 0
        END AS price_z_score,
        price_numeric / NULLIF(category_avg_price, 0) AS ratio_to_category_avg
    FROM category_stats
    WHERE category_menu_count >= 30
),
outliers AS (
    -- 최종 이상치 기준: 통계적으로 튀면서(Z-score>3) 동시에 절대적으로도 카테고리 평균의 5배 이상
    SELECT *
    FROM z_score_flagged
    WHERE ABS(price_z_score) > 3
      AND ratio_to_category_avg > 5
)
SELECT
    restaurant_id,
    restaurant_name,
    menu_name,
    main_category,
    price_numeric,
    ROUND(category_avg_price, 2) AS category_avg_price,
    ROUND(price_z_score, 2) AS price_z_score,
    ROUND(ratio_to_category_avg, 1) AS ratio_to_category_avg,
    CASE
        WHEN ABS(price_numeric / 100 - category_avg_price) / NULLIF(category_avg_price, 0) < 0.3
            THEN '소수점 오류 의심 (÷100)'
        WHEN ABS(price_numeric / 10 - category_avg_price) / NULLIF(category_avg_price, 0) < 0.3
            THEN '소수점 오류 의심 (÷10)'
        ELSE '검토 필요 (원인 불명)'
    END AS suspected_cause
FROM outliers
ORDER BY ABS(price_z_score) DESC
"""

final_price_outliers = con.execute(final_price_outliers_query).fetchdf()
print('최종 필터링 후 이상치 개수:', len(final_price_outliers))
print()
print('원인별 분류:')
print(final_price_outliers['suspected_cause'].value_counts())
print()
print('상위 20건:')
print(final_price_outliers.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

최종 필터링 후 이상치 개수: 32199

원인별 분류:
suspected_cause
검토 필요 (원인 불명)       21699
소수점 오류 의심 (÷10)     10495
소수점 오류 의심 (÷100)        5
Name: count, dtype: int64

상위 20건:
    restaurant_id                           restaurant_name                                          menu_name                  main_category  \
0           54106                             Fire Ass Thai                               Spicy Massaman Curry                   Asian Fusion   
1           19112              Tsim Yung Chinese Restaurant                               C 7. Moo Goo Gai Pan                        Chinese   
2           41586                          Taqueria Chapala                                      Mojarra Frita                        Mexican   
3           36135                         palios pizza cafe                                        Greek Bread                        Italian   
4           50119                          Nusr-Et (Dallas)                      Gold Istanbul Steak (10.5 oz)    

In [21]:
# 최종 가격 이상치에 신뢰도(confidence) 컬럼 추가
final_price_outliers['confidence_level'] = final_price_outliers['suspected_cause'].apply(
    lambda x: 'High Confidence (자동확정)' if '소수점 오류 의심' in x else 'Needs Review (수동검토)'
)

print('신뢰도별 분류:')
print(final_price_outliers['confidence_level'].value_counts())
print()
print('--- High Confidence 케이스 비율 ---')
high_conf_count = (final_price_outliers['confidence_level'] == 'High Confidence (자동확정)').sum()
print(f'{high_conf_count}건 / {len(final_price_outliers)}건 = {high_conf_count/len(final_price_outliers)*100:.1f}%')

신뢰도별 분류:
confidence_level
Needs Review (수동검토)       21699
High Confidence (자동확정)    10500
Name: count, dtype: int64

--- High Confidence 케이스 비율 ---
10500건 / 32199건 = 32.6%


In [22]:
# 최소주문금액 이상탐지
# Step 1: Internal Ratio / External Ratio 계산
min_order_ratio_query = """
WITH base AS (
    SELECT
        id AS restaurant_id,
        name AS restaurant_name,
        main_category,
        main_menu_avg_price,
        min_order_amount,
        price_data_status,
        is_synthetic_outlier,
        synthetic_outlier_type,
        -- Internal Ratio: 최소주문금액이 이 상점 메인메뉴 평균가의 몇 배인가
        min_order_amount / NULLIF(main_menu_avg_price, 0) AS internal_ratio
    FROM restaurants
    WHERE price_data_status = '정상'  -- 가격정보 미기재 상점은 비율 계산이 불가하므로 제외
),
category_avg_min_order AS (
    -- 카테고리별 평균 최소주문금액 (External Ratio 기준값)
    SELECT
        *,
        AVG(min_order_amount) OVER (PARTITION BY main_category) AS category_avg_min_order,
        COUNT(*) OVER (PARTITION BY main_category) AS category_store_count
    FROM base
)
SELECT
    *,
    -- External Ratio: 이 상점 최소주문금액이 동일 카테고리 평균의 몇 배인가
    min_order_amount / NULLIF(category_avg_min_order, 0) AS external_ratio
FROM category_avg_min_order
WHERE category_store_count >= 30  -- 표본 30건 미만 카테고리 제외
"""

min_order_ratios = con.execute(min_order_ratio_query).fetchdf()
print('계산 완료된 상점 수:', len(min_order_ratios))
print()
print('--- internal_ratio / external_ratio 기초 통계 ---')
print(min_order_ratios[['internal_ratio', 'external_ratio']].describe())

계산 완료된 상점 수: 57554

--- internal_ratio / external_ratio 기초 통계 ---
       internal_ratio  external_ratio
count    57554.000000    57554.000000
mean         1.773728        1.000000
std          0.439117        0.568978
min          0.199852        0.002488
25%          1.618925        0.678245
50%          1.750359        0.935977
75%          1.881188        1.224772
max          5.000255       21.146679


In [23]:
# Step 2: P1/P2/P3 플래깅 적용
priority_query = """
WITH base AS (
    SELECT
        id AS restaurant_id,
        name AS restaurant_name,
        main_category,
        main_menu_avg_price,
        min_order_amount,
        is_synthetic_outlier,
        synthetic_outlier_type,
        min_order_amount / NULLIF(main_menu_avg_price, 0) AS internal_ratio
    FROM restaurants
    WHERE price_data_status = '정상'
),
with_external AS (
    SELECT
        *,
        AVG(min_order_amount) OVER (PARTITION BY main_category) AS category_avg_min_order,
        COUNT(*) OVER (PARTITION BY main_category) AS category_store_count
    FROM base
),
with_ratio AS (
    SELECT
        *,
        min_order_amount / NULLIF(category_avg_min_order, 0) AS external_ratio
    FROM with_external
    WHERE category_store_count >= 30
)
SELECT
    *,
    CASE
        WHEN internal_ratio > 2.5 AND external_ratio > 1.5 THEN 'P1 (심각-진입장벽 과도)'
        WHEN internal_ratio > 1.5 AND internal_ratio <= 2.5 THEN 'P2 (경고-1인분 주문불가)'
        WHEN min_order_amount = 0 OR internal_ratio <= 0.5 THEN 'P3 (주의-과소설정/손실위험)'
        ELSE '정상'
    END AS priority
FROM with_ratio
"""

priority_result = con.execute(priority_query).fetchdf()
print('--- 우선순위별 상점 수 ---')
print(priority_result['priority'].value_counts())

--- 우선순위별 상점 수 ---
priority
P2 (경고-1인분 주문불가)     54669
P1 (심각-진입장벽 과도)       1121
정상                     896
P3 (주의-과소설정/손실위험)      868
Name: count, dtype: int64


In [24]:
print('--- internal_ratio 주요 percentile ---')
print(min_order_ratios['internal_ratio'].quantile([0.5, 0.75, 0.9, 0.95, 0.975, 0.99]))
print()
print('--- external_ratio 주요 percentile ---')
print(min_order_ratios['external_ratio'].quantile([0.5, 0.75, 0.9, 0.95, 0.975, 0.99]))
print()
# 심어둔 정답지 기준으로 실제 이상치 상점들의 internal_ratio 분포 확인
print('--- 정답지(High 이상치)의 internal_ratio 분포 ---')
print(min_order_ratios[min_order_ratios['synthetic_outlier_type'] == 'high']['internal_ratio'].describe())
print()
print('--- 정답지(Low 이상치)의 internal_ratio 분포 ---')
print(min_order_ratios[min_order_ratios['synthetic_outlier_type'] == 'low']['internal_ratio'].describe())

--- internal_ratio 주요 percentile ---
0.500    1.750359
0.750    1.881188
0.900    1.960256
0.950    1.986599
0.975    3.008715
0.990    4.147925
Name: internal_ratio, dtype: float64

--- external_ratio 주요 percentile ---
0.500    0.935977
0.750    1.224772
0.900    1.562881
0.950    1.836193
0.975    2.202572
0.990    2.909733
Name: external_ratio, dtype: float64

--- 정답지(High 이상치)의 internal_ratio 분포 ---
count    1447.000000
mean        3.966071
std         0.568290
min         3.001040
25%         3.474131
50%         3.971201
75%         4.443895
max         5.000255
Name: internal_ratio, dtype: float64

--- 정답지(Low 이상치)의 internal_ratio 분포 ---
count    1404.000000
mean        0.444787
std         0.141342
min         0.199852
25%         0.322451
50%         0.444874
75%         0.563447
max         0.699865
Name: internal_ratio, dtype: float64


In [25]:
retest_query = """
WITH base AS (
    SELECT
        id AS restaurant_id,
        name AS restaurant_name,
        main_category,
        main_menu_avg_price,
        min_order_amount,
        is_synthetic_outlier,
        synthetic_outlier_type,
        min_order_amount / NULLIF(main_menu_avg_price, 0) AS internal_ratio
    FROM restaurants
    WHERE price_data_status = '정상'
),
with_external AS (
    SELECT
        *,
        AVG(min_order_amount) OVER (PARTITION BY main_category) AS category_avg_min_order,
        COUNT(*) OVER (PARTITION BY main_category) AS category_store_count
    FROM base
),
with_ratio AS (
    SELECT
        *,
        min_order_amount / NULLIF(category_avg_min_order, 0) AS external_ratio
    FROM with_external
    WHERE category_store_count >= 30
)
SELECT
    *,
    CASE
        WHEN internal_ratio > 3.0 THEN 'P1 (심각-진입장벽 과도)'
        WHEN internal_ratio < 0.7 THEN 'P3 (주의-과소설정/손실위험)'
        WHEN internal_ratio > 1.99 THEN 'P2 (경고-회색지대)'
        ELSE '정상'
    END AS priority
FROM with_ratio
"""

priority_result = con.execute(retest_query).fetchdf()
print('--- 새 기준 우선순위별 상점 수 ---')
print(priority_result['priority'].value_counts())
print()

# 탐지 정확도(Recall) 검증: 심어둔 정답지를 SQL이 얼마나 잘 잡아냈는가
high_truth = priority_result[priority_result['synthetic_outlier_type'] == 'high']
low_truth = priority_result[priority_result['synthetic_outlier_type'] == 'low']

high_recall = (high_truth['priority'] == 'P1 (심각-진입장벽 과도)').mean()
low_recall = (low_truth['priority'] == 'P3 (주의-과소설정/손실위험)').mean()

print(f'P1 탐지 재현율(Recall): {high_recall*100:.1f}% ({(high_truth["priority"]=="P1 (심각-진입장벽 과도)").sum()}/{len(high_truth)}건)')
print(f'P3 탐지 재현율(Recall): {low_recall*100:.1f}% ({(low_truth["priority"]=="P3 (주의-과소설정/손실위험)").sum()}/{len(low_truth)}건)')

--- 새 기준 우선순위별 상점 수 ---
priority
정상                   53606
P1 (심각-진입장벽 과도)       1447
P3 (주의-과소설정/손실위험)     1404
P2 (경고-회색지대)          1097
Name: count, dtype: int64

P1 탐지 재현율(Recall): 100.0% (1447/1447건)
P3 탐지 재현율(Recall): 100.0% (1404/1404건)


In [26]:
# 이미지 품질 + 메뉴완전성 + 품절관리(KPI)
# 이미지 등록 여부: 상점별로 "이미지 있음/없음"을 확률적으로 합성 (예: 90%는 있음, 10%는 없음 — 실제 배달앱 통계에 가까운 현실적 비율)
# 메뉴/옵션 완전성: 메뉴 설명이 비어있는 비율을 실제 데이터에서 그대로 활용 (이건 합성 안 해도 됨 — menus 테이블에 실제 description 컬럼이 있음)
# 품절 관리: 상점별로 '최근 품절 메뉴 비율'을 합성

In [27]:
# 메뉴 설명(description) 결측 현황 - 실제 데이터로 '메뉴 완전성' KPI에 활용 가능한지 확인
print('--- description 결측 현황 ---')
print(food_menus['description'].isna().sum(), '/', len(food_menus))
print(f"{food_menus['description'].isna().mean()*100:.1f}% 결측")
print()

# 상점별 '설명 없는 메뉴' 비율 계산
desc_completeness = food_menus.groupby('restaurant_id').agg(
    total_menu_count=('name', 'count'),
    missing_desc_count=('description', lambda x: x.isna().sum())
).reset_index()
desc_completeness['missing_desc_ratio'] = (
    desc_completeness['missing_desc_count'] / desc_completeness['total_menu_count']
).round(3)

print('--- 상점별 설명누락비율 기초통계 ---')
print(desc_completeness['missing_desc_ratio'].describe())

--- description 결측 현황 ---
1215336 / 4236129
28.7% 결측

--- 상점별 설명누락비율 기초통계 ---
count    59376.000000
mean         0.300643
std          0.290931
min          0.000000
25%          0.054000
50%          0.220000
75%          0.463000
max          1.000000
Name: missing_desc_ratio, dtype: float64


In [28]:
np.random.seed(42)

# 1. 이미지 등록 여부 합성
#    실제 배달앱 통계상 대형 프랜차이즈/체인은 이미지 등록률이 높고, 소규모 개인 상점은 낮은 경향이 있음을 반영
#    -> ratings(리뷰수)가 많을수록(=자리잡은 상점일수록) 이미지 등록 확률이 높다고 가정 (간이 규칙)
analysis_restaurants['ratings_filled'] = analysis_restaurants['ratings'].fillna(0)

# 리뷰수 기반 이미지 등록 확률 (리뷰 많을수록 이미지 등록 확률 ↑, 최소 70% ~ 최대 97%)
image_prob = 0.70 + 0.27 * (analysis_restaurants['ratings_filled'].rank(pct=True))
analysis_restaurants['has_image'] = np.random.rand(len(analysis_restaurants)) < image_prob

# 2. 이미지가 있는 상점 중 일부는 저해상도로 합성 (있음/저품질/없음 3단계)
low_res_mask = analysis_restaurants['has_image'] & (np.random.rand(len(analysis_restaurants)) < 0.08)
analysis_restaurants['image_status'] = np.select(
    [~analysis_restaurants['has_image'], low_res_mask],
    ['이미지 없음', '저해상도'],
    default='정상'
)

# -> 다른 지표(리뷰수)에 연동된 확률

# 3. 품절 관리 - 최근 품절 메뉴 비율 (0~15% 범위에서 랜덤, 일부 상점은 관리 소홀로 높은 비율)
soldout_ratio = np.random.beta(2, 20, size=len(analysis_restaurants))  # 대부분 낮은 값에 몰리는 분포
analysis_restaurants['soldout_menu_ratio'] = soldout_ratio.round(3)

print('--- 이미지 상태 분포 ---')
print(analysis_restaurants['image_status'].value_counts())
print()
print('--- 품절 메뉴 비율 기초통계 ---')
print(analysis_restaurants['soldout_menu_ratio'].describe())
# -> 처음부터 상점마다 랜덤한 비율 값 자체를 부여

--- 이미지 상태 분포 ---
image_status
정상        45596
이미지 없음     9761
저해상도       4019
Name: count, dtype: int64

--- 품절 메뉴 비율 기초통계 ---
count    59376.000000
mean         0.090968
std          0.060268
min          0.000000
25%          0.046000
50%          0.079000
75%          0.123000
max          0.513000
Name: soldout_menu_ratio, dtype: float64


In [29]:
# 메뉴 완전성(description 누락비율)을 restaurants 테이블에 합쳐서 최종 KPI 원본 데이터 완성
analysis_restaurants = analysis_restaurants.merge(
    desc_completeness[['restaurant_id', 'missing_desc_ratio']],
    left_on='id', right_on='restaurant_id', how='left'
).drop(columns=['restaurant_id'])

print('--- 최종 KPI 원본 필드 확인 ---')
print(analysis_restaurants[['name', 'main_menu_avg_price', 'min_order_amount',
                              'image_status', 'soldout_menu_ratio', 'missing_desc_ratio']].head(5))

--- 최종 KPI 원본 필드 확인 ---
                                             name  main_menu_avg_price  min_order_amount image_status  soldout_menu_ratio  missing_desc_ratio
0               PJ Fresh (224 Daniel Payne Drive)             3.007391              5.07           정상               0.048               0.242
1                  J' ti`'z Smoothie-N-Coffee Bar             5.391724             10.65       이미지 없음               0.061               0.233
2  Philly Fresh Cheesesteaks (541-B Graymont Ave)            12.580000             42.91           정상               0.161               0.179
3         Papa Murphy's (1580 Montgomery Highway)            11.546667             20.78           정상               0.009               0.027
4                Nelson Brothers Cafe (17th St N)             3.838889              6.06           정상               0.045               0.591


In [34]:
# 필수정보 누락 체크: 상점명, 주소, 카테고리 등 기본 정보 결측 여부
analysis_restaurants['missing_basic_info'] = (
    analysis_restaurants['name'].isna() |
    analysis_restaurants['full_address'].isna() |
    analysis_restaurants['main_category'].isna() |
    (analysis_restaurants['name'].str.strip() == '')
)

print('--- 필수정보 누락 상점 수 ---')
print(analysis_restaurants['missing_basic_info'].value_counts())

--- 필수정보 누락 상점 수 ---
missing_basic_info
False    58913
True       463
Name: count, dtype: int64


In [36]:
# 가격 이상치를 '상점 단위' 지표로 집계

# 상점별로 가격 이상치 메뉴가 몇 개, 몇 %나 있는지 집계
price_issue_summary = final_price_outliers.groupby('restaurant_id').agg(
    price_outlier_count=('menu_name', 'count')
).reset_index()

# 상점별 전체 메뉴 수 대비 이상치 비율 계산
menu_count_per_store = food_menus.groupby('restaurant_id').size().rename('total_menu_count').reset_index()

price_issue_summary = price_issue_summary.merge(menu_count_per_store, on='restaurant_id', how='left')
price_issue_summary['price_outlier_ratio'] = (
    price_issue_summary['price_outlier_count'] / price_issue_summary['total_menu_count']
).round(4)

print('--- 상점별 가격 이상치 비율 기초통계 ---')
print(price_issue_summary['price_outlier_ratio'].describe())
print()
print('이상치가 하나라도 있는 상점 수:', len(price_issue_summary), '/', len(analysis_restaurants))

# analysis_restaurants에 병합 (이상치가 없는 상점은 0으로 처리)
analysis_restaurants = analysis_restaurants.merge(
    price_issue_summary[['restaurant_id', 'price_outlier_ratio']],
    left_on='id', right_on='restaurant_id', how='left'
).drop(columns=['restaurant_id'])

analysis_restaurants['price_outlier_ratio'] = analysis_restaurants['price_outlier_ratio'].fillna(0)

print('--- 최종 병합 확인 ---')
print(analysis_restaurants['price_outlier_ratio'].describe())

--- 상점별 가격 이상치 비율 기초통계 ---
count    8789.000000
mean        0.052969
std         0.071921
min         0.001900
25%         0.016900
50%         0.034000
75%         0.061200
max         1.000000
Name: price_outlier_ratio, dtype: float64

이상치가 하나라도 있는 상점 수: 8789 / 59376
--- 최종 병합 확인 ---
count    59376.000000
mean         0.007841
std          0.033458
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: price_outlier_ratio, dtype: float64


In [38]:
# internal_ratio를 analysis_restaurants에 추가 (이미 있는 두 컬럼으로 계산 가능)
analysis_restaurants['internal_ratio'] = (
    analysis_restaurants['min_order_amount'] / analysis_restaurants['main_menu_avg_price']
)

print('--- internal_ratio 확인 ---')
print(analysis_restaurants['internal_ratio'].describe())

--- internal_ratio 확인 ---
count    58681.000000
mean         1.773798
std          0.439483
min          0.199852
25%          1.618921
50%          1.750358
75%          1.881212
max          5.000255
Name: internal_ratio, dtype: float64


In [ ]:
# Catalog Health Score 산출 (모든 KPI 종합 0~100점)

In [40]:
df = analysis_restaurants.copy()

# 1. 가격 정확성 점수 (30%): 이상치 비율이 높을수록 감점. 비율 20% 이상이면 0점 처리
df['score_price'] = (1 - (df['price_outlier_ratio'] / 0.2).clip(0, 1)) * 100

# 2. 최소주문금액 정합성 점수 (25%): internal_ratio가 정상범위(0.7~2.0)에서 멀어질수록 감점
def min_order_score(ratio):
    if pd.isna(ratio):
        return 50  # 계산 불가 상점은 중립값
    if 0.7 <= ratio <= 2.0:
        return 100
    elif ratio > 2.0:
        return max(0, 100 - (ratio - 2.0) * 40)
    else:
        return max(0, 100 - (0.7 - ratio) * 140)

df['score_min_order'] = df['internal_ratio'].apply(min_order_score)

# 3. 이미지 품질 점수 (25%)
image_score_map = {'정상': 100, '저해상도': 50, '이미지 없음': 0}
df['score_image'] = df['image_status'].map(image_score_map)

# 4. 메뉴/옵션 완전성 점수 (10%)
df['score_menu_completeness'] = (1 - df['missing_desc_ratio']) * 100

# 5. 품절 관리 점수 (5%)
df['score_soldout'] = (1 - (df['soldout_menu_ratio'] / 0.3).clip(0, 1)) * 100

# 6. 필수정보 누락 점수 (5%)
df['score_basic_info'] = np.where(df['missing_basic_info'], 0, 100)

# 최종 가중합산 -> Catalog Health Score
df['catalog_health_score'] = (
    df['score_price'] * 0.30 +
    df['score_min_order'] * 0.25 +
    df['score_image'] * 0.25 +
    df['score_menu_completeness'] * 0.10 +
    df['score_soldout'] * 0.05 +
    df['score_basic_info'] * 0.05
).round(1)

analysis_restaurants = df

print('--- Catalog Health Score 기초통계 ---')
print(analysis_restaurants['catalog_health_score'].describe())

--- Catalog Health Score 기초통계 ---
count    59376.000000
mean        88.572667
std         11.154990
min         20.200000
25%         83.700000
50%         93.300000
75%         96.900000
max        100.000000
Name: catalog_health_score, dtype: float64


In [41]:
# 등급 나누기 + 최하위 상점 확인

# 점수 구간별 등급 분류
bins = [0, 40, 60, 80, 100]
labels = ['위험(0-40)', '경고(40-60)', '양호(60-80)', '우수(80-100)']
analysis_restaurants['health_grade'] = pd.cut(
    analysis_restaurants['catalog_health_score'], bins=bins, labels=labels, include_lowest=True
)

print('--- 등급별 상점 분포 ---')
print(analysis_restaurants['health_grade'].value_counts().sort_index())

--- 등급별 상점 분포 ---
health_grade
위험(0-40)         37
경고(40-60)       802
양호(60-80)     12030
우수(80-100)    46507
Name: count, dtype: int64


In [42]:
# 최하위 10개 상점 - 실제로 어떤 문제들이 겹쳐있는지 확인
worst_stores = analysis_restaurants.nsmallest(10, 'catalog_health_score')[
    ['name', 'catalog_health_score', 'score_price', 'score_min_order',
     'score_image', 'score_menu_completeness', 'score_soldout', 'score_basic_info']
]
print('--- 최하위 10개 상점 ---')
print(worst_stores)

--- 최하위 10개 상점 ---
                                              name  catalog_health_score  score_price  score_min_order  score_image  score_menu_completeness  \
49983                                     Bar 5015                  20.2         0.00        11.827715            0                    100.0   
42954                 Mastro's (9595 Six Pines Dr)                  22.2         0.00        21.326148            0                     80.4   
35526           Del Frisco's Double (812 Main St.)                  27.1         0.00         1.130578           50                     46.4   
39520                   Fajita Pete's (Clear Lake)                  28.5         3.85        51.125425            0                     76.9   
19090                               Olivia Macaron                  29.9         0.00         0.000000           50                    100.0   
43236  Morton's The Steakhouse  (25 Waterway Ave.)                  31.2         0.00       100.000000            0  

In [43]:
# High Confidence(자동확정)만 필터링해서 상점별 비율 재계산
high_conf_outliers = final_price_outliers[final_price_outliers['confidence_level'] == 'High Confidence (자동확정)']

price_issue_summary_v2 = high_conf_outliers.groupby('restaurant_id').agg(
    price_outlier_count=('menu_name', 'count')
).reset_index()

price_issue_summary_v2 = price_issue_summary_v2.merge(menu_count_per_store, on='restaurant_id', how='left')
price_issue_summary_v2['price_outlier_ratio_v2'] = (
    price_issue_summary_v2['price_outlier_count'] / price_issue_summary_v2['total_menu_count']
).round(4)

# analysis_restaurants에 재병합
analysis_restaurants = analysis_restaurants.merge(
    price_issue_summary_v2[['restaurant_id', 'price_outlier_ratio_v2']],
    left_on='id', right_on='restaurant_id', how='left'
).drop(columns=['restaurant_id'])
analysis_restaurants['price_outlier_ratio_v2'] = analysis_restaurants['price_outlier_ratio_v2'].fillna(0)

print('--- 재계산된 가격 이상치 비율 (High Confidence만) ---')
print(analysis_restaurants['price_outlier_ratio_v2'].describe())
print()
print('이상치 있는 상점 수:', (analysis_restaurants['price_outlier_ratio_v2'] > 0).sum())

--- 재계산된 가격 이상치 비율 (High Confidence만) ---
count    59376.000000
mean         0.002594
std          0.017473
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: price_outlier_ratio_v2, dtype: float64

이상치 있는 상점 수: 4222


In [44]:
# score_price를 High Confidence 기준(price_outlier_ratio_v2)으로 재계산
analysis_restaurants['score_price'] = (
    1 - (analysis_restaurants['price_outlier_ratio_v2'] / 0.2).clip(0, 1)
) * 100

# Catalog Health Score 재산출 (다른 5개 KPI 점수는 그대로 유지, score_price만 교체)
analysis_restaurants['catalog_health_score'] = (
    analysis_restaurants['score_price'] * 0.30 +
    analysis_restaurants['score_min_order'] * 0.25 +
    analysis_restaurants['score_image'] * 0.25 +
    analysis_restaurants['score_menu_completeness'] * 0.10 +
    analysis_restaurants['score_soldout'] * 0.05 +
    analysis_restaurants['score_basic_info'] * 0.05
).round(1)

print('--- 재산출된 Catalog Health Score 기초통계 ---')
print(analysis_restaurants['catalog_health_score'].describe())

--- 재산출된 Catalog Health Score 기초통계 ---
count    59376.000000
mean        89.285292
std         10.747214
min         22.200000
25%         85.100000
50%         94.000000
75%         97.100000
max        100.000000
Name: catalog_health_score, dtype: float64


In [45]:
# 등급 재분류
analysis_restaurants['health_grade'] = pd.cut(
    analysis_restaurants['catalog_health_score'], bins=bins, labels=labels, include_lowest=True
)
print('--- 등급별 상점 분포 (재산출) ---')
print(analysis_restaurants['health_grade'].value_counts().sort_index())

--- 등급별 상점 분포 (재산출) ---
health_grade
위험(0-40)         15
경고(40-60)       514
양호(60-80)     11471
우수(80-100)    47376
Name: count, dtype: int64


In [46]:
# 최하위 10개 상점 재확인 - 스테이크하우스 같은 오탐이 사라졌는지 체크
worst_stores_v2 = analysis_restaurants.nsmallest(10, 'catalog_health_score')[
    ['name', 'catalog_health_score', 'score_price', 'score_min_order',
     'score_image', 'score_menu_completeness', 'score_soldout', 'score_basic_info']
]
print('--- 최하위 10개 상점 (재산출) ---')
print(worst_stores_v2)

--- 최하위 10개 상점 (재산출) ---
                                              name  catalog_health_score  score_price  score_min_order  score_image  score_menu_completeness  \
42954                 Mastro's (9595 Six Pines Dr)                  22.2         0.00        21.326148            0                     80.4   
43236  Morton's The Steakhouse  (25 Waterway Ave.)                  31.2         0.00       100.000000            0                      0.0   
54278                   Palmeras Tex-Mex Con Sabor                  33.7         0.00       100.000000            0                      0.0   
27280       Rabble-Rouser Chocolate &amp; Craft Co                  33.8         0.00       100.000000            0                      0.0   
42711                  BIG JACK’S BURGERS AND MORE                  35.7       100.00         0.550649            0                     11.5   
49983                                     Bar 5015                  36.5        54.55        11.827715         

In [47]:
# High Confidence를 '÷100 소수점 오류'로만 한정 (÷10은 오탐 위험이 높으므로 제외)
high_conf_v3 = final_price_outliers[final_price_outliers['suspected_cause'] == '소수점 오류 의심 (÷100)']

print('÷100만 필터링한 건수:', len(high_conf_v3))

price_issue_summary_v3 = high_conf_v3.groupby('restaurant_id').agg(
    price_outlier_count=('menu_name', 'count')
).reset_index()

price_issue_summary_v3 = price_issue_summary_v3.merge(menu_count_per_store, on='restaurant_id', how='left')
price_issue_summary_v3['price_outlier_ratio_v3'] = (
    price_issue_summary_v3['price_outlier_count'] / price_issue_summary_v3['total_menu_count']
).round(4)

analysis_restaurants = analysis_restaurants.merge(
    price_issue_summary_v3[['restaurant_id', 'price_outlier_ratio_v3']],
    left_on='id', right_on='restaurant_id', how='left'
).drop(columns=['restaurant_id'])
analysis_restaurants['price_outlier_ratio_v3'] = analysis_restaurants['price_outlier_ratio_v3'].fillna(0)

# score_price 재재계산
analysis_restaurants['score_price'] = (
    1 - (analysis_restaurants['price_outlier_ratio_v3'] / 0.2).clip(0, 1)
) * 100

# Health Score 재산출
analysis_restaurants['catalog_health_score'] = (
    analysis_restaurants['score_price'] * 0.30 +
    analysis_restaurants['score_min_order'] * 0.25 +
    analysis_restaurants['score_image'] * 0.25 +
    analysis_restaurants['score_menu_completeness'] * 0.10 +
    analysis_restaurants['score_soldout'] * 0.05 +
    analysis_restaurants['score_basic_info'] * 0.05
).round(1)

worst_stores_v3 = analysis_restaurants.nsmallest(10, 'catalog_health_score')[
    ['name', 'catalog_health_score', 'score_price', 'score_min_order', 'score_image']
]
print(worst_stores_v3)

÷100만 필터링한 건수: 5
                                                    name  catalog_health_score  score_price  score_min_order  score_image
42711                        BIG JACK’S BURGERS AND MORE                  35.7        100.0         0.550649            0
33607                     Mariscos La Marina (Lancaster)                  37.2        100.0         1.320615            0
46137                                   Sabor De Sanchez                  38.4        100.0         0.000000            0
50980                         Treebeards (711 Louisiana)                  38.8        100.0         0.000000            0
56756  Hawaii Poke &amp; Ramen (Corpus Christi, SPI Drv)                  38.9        100.0         0.000000            0
38008                            Fire Pizza (Austin, TX)                  39.2        100.0         0.000000            0
29403                  Cafe Rio (1205 North Main Street)                  39.7        100.0         7.839885            0
40778  

In [48]:
# 등급 재분류 (최종 버전)
analysis_restaurants['health_grade'] = pd.cut(
    analysis_restaurants['catalog_health_score'], bins=bins, labels=labels, include_lowest=True
)
print('--- 등급별 상점 분포 (최종) ---')
print(analysis_restaurants['health_grade'].value_counts().sort_index())
print()
print('--- 비율(%) ---')
print((analysis_restaurants['health_grade'].value_counts(normalize=True).sort_index() * 100).round(1))

--- 등급별 상점 분포 (최종) ---
health_grade
위험(0-40)          8
경고(40-60)       402
양호(60-80)     11276
우수(80-100)    47690
Name: count, dtype: int64

--- 비율(%) ---
health_grade
위험(0-40)       0.0
경고(40-60)      0.7
양호(60-80)     19.0
우수(80-100)    80.3
Name: proportion, dtype: float64


In [49]:
# Business Impact 분석 (품질 낮은 그룹 vs 높은 그룹 성과 차이)
# -> 카탈로그 품질이 낮으면 실제로 비즈니스 성과(평점/리뷰수)에서 차이가 나는가?

In [50]:
# Step 1: Health Grade별 평점/리뷰수 비교
business_impact_query = """
SELECT
    health_grade,
    COUNT(*) AS store_count,
    COUNT(score) AS store_with_score,  -- score가 있는 상점 수 (결측 제외 확인용)
    ROUND(AVG(score), 3) AS avg_score,
    ROUND(AVG(ratings), 1) AS avg_ratings,
    ROUND(MEDIAN(ratings), 1) AS median_ratings
FROM restaurants
GROUP BY health_grade
ORDER BY
    CASE health_grade
        WHEN '위험(0-40)' THEN 1
        WHEN '경고(40-60)' THEN 2
        WHEN '양호(60-80)' THEN 3
        WHEN '우수(80-100)' THEN 4
    END
"""

# health_grade가 담긴 최신 analysis_restaurants를 duckdb에 재등록
con.register('restaurants', analysis_restaurants)

business_impact = con.execute(business_impact_query).fetchdf()
print(business_impact)

  health_grade  store_count  store_with_score  avg_score  avg_ratings  median_ratings
0     위험(0-40)            8                 3      4.633         38.7            41.0
1    경고(40-60)          402               153      4.539         52.9            33.0
2    양호(60-80)        11276              4612      4.532         57.4            38.0
3   우수(80-100)        47690             29386      4.548         78.8            56.0


In [51]:
# 표본이 너무 작은 '위험' 등급 대신, 연속형 변수(catalog_health_score)와 ratings의 상관관계를 봐보기
correlation_query = """
SELECT
    CORR(catalog_health_score, score) AS corr_score,
    CORR(catalog_health_score, ratings) AS corr_ratings
FROM restaurants
WHERE score IS NOT NULL
"""
print(con.execute(correlation_query).fetchdf())

   corr_score  corr_ratings
0    0.022793      0.096599


In [53]:
# health_score를 10점 구간으로 나눠서 ratings 평균 추이를 더 세밀하게 확인
detailed_query = """
SELECT
    FLOOR(catalog_health_score / 10) * 10 AS score_bucket,
    COUNT(*) AS store_count,
    ROUND(AVG(ratings), 1) AS avg_ratings,
    ROUND(AVG(score), 3) AS avg_score
FROM restaurants
WHERE ratings IS NOT NULL
GROUP BY score_bucket
ORDER BY score_bucket
"""
print(con.execute(detailed_query).fetchdf())

# => Catalog Health Score와 리뷰수/평점 사이에는 뚜렷한 선형 관계가 없다
# 80점 미만 구간에서는 점수와 관계없이 리뷰수가 비슷하게 낮은 수준에 머물다가, 80점을 넘는 순간부터 뚜렷하게 높아지는 임계점 패턴

   score_bucket  store_count  avg_ratings  avg_score
0          30.0            3         38.7      4.633
1          40.0           37         55.1      4.505
2          50.0          113         52.7      4.561
3          60.0         1313         53.6      4.519
4          70.0         3290         58.8      4.537
5          80.0         4544         75.3      4.547
6          90.0        24842         79.4      4.548
7         100.0           12         49.0      4.383


In [32]:
# P1/P2/P3 최종 우선순위 도출

In [54]:
# 설계 방향: Severity × Business Impact × 개선 가능성
# 리뷰수(=이 상점의 노출/트래픽 규모)를 성과 예측 지표가 아니라 '영향 범위' 지표로 재해석

priority_final_query = """
WITH scored AS (
    SELECT
        id, name, health_grade, catalog_health_score,
        ratings, score_price, score_min_order, score_image,
        score_menu_completeness, score_soldout, score_basic_info,
        -- 영향 범위: 리뷰수 상위 25% 여부 (결측은 미노출 상점으로 간주, 낮은 영향)
        CASE WHEN ratings >= (SELECT QUANTILE_CONT(ratings, 0.75) FROM restaurants WHERE ratings IS NOT NULL)
             THEN 1 ELSE 0 END AS high_exposure,
        -- 개선 용이성: 이미지/가격 문제는 즉시조치 가능(쉬움), 최소주문금액만 문제면 협의 필요(어려움)
        CASE
            WHEN score_image < 50 OR score_price < 50 THEN '쉬움 (즉시조치)'
            WHEN score_min_order < 50 THEN '어려움 (상점주 협의 필요)'
            ELSE '보통'
        END AS improvement_feasibility
    FROM restaurants
)
SELECT
    *,
    CASE
        WHEN health_grade IN ('위험(0-40)', '경고(40-60)') AND high_exposure = 1 THEN 'P1'
        WHEN health_grade IN ('위험(0-40)', '경고(40-60)') AND high_exposure = 0 THEN 'P2'
        WHEN health_grade = '양호(60-80)' AND high_exposure = 1 THEN 'P2'
        ELSE 'P3'
    END AS final_priority
FROM scored
"""

priority_final = con.execute(priority_final_query).fetchdf()
print('--- 최종 우선순위 분포 ---')
print(priority_final['final_priority'].value_counts())
print()
print('--- 우선순위별 개선 용이성 분포 ---')
print(priority_final.groupby('final_priority')['improvement_feasibility'].value_counts())

--- 최종 우선순위 분포 ---
final_priority
P3    58258
P2     1095
P1       23
Name: count, dtype: int64

--- 우선순위별 개선 용이성 분포 ---
final_priority  improvement_feasibility
P1              쉬움 (즉시조치)                     18
                어려움 (상점주 협의 필요)                5
P2              쉬움 (즉시조치)                    795
                어려움 (상점주 협의 필요)              192
                보통                           108
P3              보통                         48090
                쉬움 (즉시조치)                   8948
                어려움 (상점주 협의 필요)             1220
Name: count, dtype: int64


In [56]:
# P1 상점 리스트 미리보기 - 실제 액션 리포트에 들어갈 형태
p1_stores = priority_final[priority_final['final_priority'] == 'P1'].sort_values('catalog_health_score')[
    ['name', 'catalog_health_score', 'ratings', 'score_price', 'score_min_order', 'score_image', 'improvement_feasibility']
]
print('--- P1 (최우선 조치) 상점 미리보기 ---')
print(p1_stores.head(15))
print()
print('P1 상점 총 개수:', len(p1_stores))

# => 59,376개 상점 중, 리뷰수 기준 상위 25%(고영향)이면서 카탈로그 관리가 부실한(위험/경고 등급) 23개 상점을 P1으로 압축했다. 이 중 78%는 이미지 등록만으로 즉시 개선 가능하다

--- P1 (최우선 조치) 상점 미리보기 ---
                                              name  catalog_health_score  ratings  score_price  score_min_order  score_image improvement_feasibility
19471     Chipotle Mexican Grill (6230 Rolling Rd)                  43.7    147.0        100.0         1.610127            0               쉬움 (즉시조치)
11177                               Khao Moo Dang                   45.5    128.0        100.0         3.283333            0               쉬움 (즉시조치)
40165                                   Take a Bao                  46.1    200.0        100.0         0.000000            0               쉬움 (즉시조치)
18090                        Rasika (Penn Quarter)                  46.4    275.0        100.0         0.000000            0               쉬움 (즉시조치)
54613               McDonald's® (Lakeview Parkway)                  48.9    147.0        100.0        34.950550            0               쉬움 (즉시조치)
10714                 Nom Nom Restaurant and Grill                  50.5    12

In [33]:
# 대시보드용 데이터 export

In [57]:
# Step 1: 최종 우선순위 결과를 analysis_restaurants에 병합

# priority_final의 핵심 컬럼만 병합
analysis_restaurants = analysis_restaurants.merge(
    priority_final[['id', 'high_exposure', 'improvement_feasibility', 'final_priority']],
    on='id', how='left'
)

print('병합 확인:')
print(analysis_restaurants['final_priority'].value_counts())

병합 확인:
final_priority
P3    58258
P2     1095
P1       23
Name: count, dtype: int64


In [58]:
# Step 2: 대시보드용 최종 테이블 만들기 (컬럼명 정리)

# Tableau/Power BI 대시보드용 - 상점 단위 최종 테이블
dashboard_export = analysis_restaurants[[
    'id', 'name', 'main_category', 'full_address', 'zip_code', 'lat', 'lng',
    'score', 'ratings',
    'main_menu_avg_price', 'min_order_amount', 'internal_ratio',
    'image_status', 'soldout_menu_ratio', 'missing_desc_ratio', 'missing_basic_info',
    'score_price', 'score_min_order', 'score_image',
    'score_menu_completeness', 'score_soldout', 'score_basic_info',
    'catalog_health_score', 'health_grade',
    'high_exposure', 'improvement_feasibility', 'final_priority'
]].copy()

# Tableau에서 바로 알아보기 쉽게 컬럼명 한글/영문 정리
dashboard_export = dashboard_export.rename(columns={
    'id': 'restaurant_id',
    'name': 'restaurant_name',
    'score': 'avg_rating',
    'ratings': 'review_count',
    'catalog_health_score': 'health_score',
    'final_priority': 'action_priority'
})

print('최종 export 테이블 shape:', dashboard_export.shape)
print(dashboard_export.head(3))

최종 export 테이블 shape: (59376, 27)
   restaurant_id                                 restaurant_name   main_category                                       full_address zip_code  \
0              1               PJ Fresh (224 Daniel Payne Drive)         Burgers      224 Daniel Payne Drive, Birmingham, AL, 35207    35207   
1              2                  J' ti`'z Smoothie-N-Coffee Bar  Coffee and Tea  1521 Pinson Valley Parkway, Birmingham, AL, 35217    35217   
2              3  Philly Fresh Cheesesteaks (541-B Graymont Ave)        American          541-B Graymont Ave, Birmingham, AL, 35204    35204   

         lat        lng  avg_rating  review_count  main_menu_avg_price  min_order_amount  internal_ratio image_status  soldout_menu_ratio  \
0  33.562365 -86.830703         NaN           NaN             3.007391              5.07        1.685846           정상               0.048   
1  33.583640 -86.773330         NaN           NaN             5.391724             10.65        1.975249    

In [59]:
# Step 3: 메뉴 단위 가격 이상치 상세

# 상점 클릭 시 "어떤 메뉴가 문제인지" 드릴다운할 수 있는 상세 테이블
price_outlier_detail_export = final_price_outliers[[
    'restaurant_id', 'restaurant_name', 'menu_name', 'main_category',
    'price_numeric', 'category_avg_price', 'price_z_score',
    'ratio_to_category_avg', 'suspected_cause', 'confidence_level'
]].rename(columns={'price_numeric': 'menu_price'})

print('가격 이상치 상세 테이블 shape:', price_outlier_detail_export.shape)
print(price_outlier_detail_export.head(3))

가격 이상치 상세 테이블 shape: (32199, 10)
   restaurant_id               restaurant_name             menu_name main_category  menu_price  category_avg_price  price_z_score  \
0          54106                 Fire Ass Thai  Spicy Massaman Curry  Asian Fusion      1395.0               11.98         122.52   
1          19112  Tsim Yung Chinese Restaurant  C 7. Moo Goo Gai Pan       Chinese       925.0               11.97         117.18   
2          41586              Taqueria Chapala         Mojarra Frita       Mexican      1049.0                9.87          98.97   

   ratio_to_category_avg   suspected_cause        confidence_level  
0                  116.4  소수점 오류 의심 (÷100)  High Confidence (자동확정)  
1                   77.3  소수점 오류 의심 (÷100)  High Confidence (자동확정)  
2                  106.3  소수점 오류 의심 (÷100)  High Confidence (자동확정)  


In [60]:
# Step 4: CSV로 저장 + 다운로드

from google.colab import files

dashboard_export.to_csv('catalog_triage_dashboard.csv', index=False, encoding='utf-8-sig')
price_outlier_detail_export.to_csv('price_outlier_details.csv', index=False, encoding='utf-8-sig')

print('저장 완료:')
print('- catalog_triage_dashboard.csv:', dashboard_export.shape)
print('- price_outlier_details.csv:', price_outlier_detail_export.shape)

files.download('catalog_triage_dashboard.csv')
files.download('price_outlier_details.csv')

저장 완료:
- catalog_triage_dashboard.csv: (59376, 27)
- price_outlier_details.csv: (32199, 10)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>